In [6]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.metrics import confusion_matrix

# ============================================================
# CONFIG
# ============================================================

LABELS = ["bug", "feature", "question"]

ZERO_SHOT_DIR = Path("../rqs/zero_shot")
OUTPUT_DIR = Path("./confusion_figures")
OUTPUT_DIR.mkdir(exist_ok=True)

REPOS = [
    "facebook_react",
    "bitcoin_bitcoin",
    "opencv_opencv",
    "tensorflow_tensorflow",
    "microsoft_vscode",
]

MODELS = [
    "Qwen/Qwen2.5-7B-Instruct",
    "google/gemma-3-12b-it",
    "meta-llama/Llama-3.1-8B-Instruct",
    "mistralai/Mistral-7B-Instruct-v0.3",
]

MODEL_DISPLAY = {
    "Qwen/Qwen2.5-7B-Instruct": "Qwen2.5-7B",
    "google/gemma-3-12b-it": "Gemma-3-12B",
    "meta-llama/Llama-3.1-8B-Instruct": "Llama-3.1-8B",
    "mistralai/Mistral-7B-Instruct-v0.3": "Mistral-7B",
}

# ============================================================
# LOAD PREDICTIONS
# ============================================================

rows = []

for repo in REPOS:
    for model in MODELS:

        provider, model_name = model.split("/", 1)

        pred_dir = (
            ZERO_SHOT_DIR
            / repo
            / provider
            / model_name
            / "logs"
            / "generations"
            / "zero_shot"
            / "run_0"
        )

        if not pred_dir.exists():
            continue

        for file in pred_dir.glob("predictions_*.csv"):

            df = pd.read_csv(file)
            df["Repo"] = repo
            df["Model"] = model

            rows.append(df)

pred_df = pd.concat(rows, ignore_index=True)

# ============================================================
# HEATMAP STYLE
# ============================================================

sns.set_style("white")
plt.rcParams["font.size"] = 18

# ============================================================
# GENERATE HEATMAP FIGURES
# ============================================================

for repo in REPOS:

    fig, axes = plt.subplots(
        1,
        len(MODELS),
        figsize=(20, 5),
        constrained_layout=True
    )

    for i, model in enumerate(MODELS):

        subset = pred_df[
            (pred_df["Repo"] == repo) &
            (pred_df["Model"] == model)
        ]

        if subset.empty:
            axes[i].axis("off")
            continue

        # Confusion matrix
        cm = confusion_matrix(
            subset["Ground_Truth"],
            subset["Prediction"],
            labels=LABELS
        )

        # Normalize rows
        cm_norm = cm.astype("float") / cm.sum(axis=1)[:, None]

        sns.heatmap(
            cm_norm,
            annot=cm,
            fmt="d",
            cmap="Blues",
            cbar=False,
            xticklabels=LABELS,
            yticklabels=LABELS,
            ax=axes[i],
            linewidths=0.5
        )

        axes[i].set_title(MODEL_DISPLAY.get(model, model))
        axes[i].set_xlabel("Predicted")
        axes[i].set_ylabel("True")

    # ============================================================
    # SAVE FIGURE
    # ============================================================

    output_file = OUTPUT_DIR / f"confusion_{repo}.pdf"

    plt.savefig(
        output_file,
        bbox_inches="tight",
        dpi=300
    )

    plt.close()

    print(f"Saved: {output_file}")

Saved: confusion_figures\confusion_facebook_react.pdf
Saved: confusion_figures\confusion_bitcoin_bitcoin.pdf
Saved: confusion_figures\confusion_opencv_opencv.pdf
Saved: confusion_figures\confusion_tensorflow_tensorflow.pdf
Saved: confusion_figures\confusion_microsoft_vscode.pdf


In [1]:
import pandas as pd
from pathlib import Path
from sklearn.metrics import classification_report

# ============================================================
# CONFIG
# ============================================================

ZERO_SHOT_BASE_DIR = Path("../rqs/zero_shot")

REPOS = [
    "facebook_react",
    "bitcoin_bitcoin",
    "opencv_opencv",
    "tensorflow_tensorflow",
    "microsoft_vscode",
]

MODELS = [
    "google/gemma-3-12b-it",
    "meta-llama/Llama-3.1-8B-Instruct",
    "mistralai/Mistral-7B-Instruct-v0.3",
    "Qwen/Qwen2.5-7B-Instruct",
]

LABELS = ["bug", "feature", "question"]

# ============================================================
# COMPUTE RESULTS
# ============================================================

results = {}

for repo in REPOS:

    results[repo] = {}

    for model in MODELS:

        provider, model_name = model.split("/", 1)

        run_dir = (
            ZERO_SHOT_BASE_DIR
            / repo
            / provider
            / model_name
            / "logs"
            / "generations"
            / "zero_shot"
            / "run_0"
        )

        if not run_dir.exists():
            continue

        pred_files = list(run_dir.glob("predictions_*.csv"))

        if not pred_files:
            continue

        df = pd.read_csv(pred_files[0])

        if not {"Ground_Truth", "Prediction"}.issubset(df.columns):
            continue

        # ----------------------------------------------------
        # Compute classification report
        # ----------------------------------------------------

        report = classification_report(
            df["Ground_Truth"],
            df["Prediction"],
            labels=LABELS,
            output_dict=True,
            zero_division=0
        )

        results[repo][model] = {}

        for label in LABELS:

            results[repo][model][label] = {
                "precision": report[label]["precision"],
                "recall": report[label]["recall"],
                "f1-score": report[label]["f1-score"],
                "support": report[label]["support"],
            }

# ============================================================
# PRINT RESULTS DICT
# ============================================================

results

{'facebook_react': {'google/gemma-3-12b-it': {'bug': {'precision': 0.5625,
    'recall': 1.0,
    'f1-score': 0.72,
    'support': 18.0},
   'feature': {'precision': 0.8,
    'recall': 0.5,
    'f1-score': 0.6153846153846154,
    'support': 8.0},
   'question': {'precision': 0.8333333333333334,
    'recall': 0.43478260869565216,
    'f1-score': 0.5714285714285714,
    'support': 23.0}},
  'meta-llama/Llama-3.1-8B-Instruct': {'bug': {'precision': 0.3695652173913043,
    'recall': 0.9444444444444444,
    'f1-score': 0.53125,
    'support': 18.0},
   'feature': {'precision': 0.5,
    'recall': 0.125,
    'f1-score': 0.2,
    'support': 8.0},
   'question': {'precision': 1.0,
    'recall': 0.043478260869565216,
    'f1-score': 0.08333333333333333,
    'support': 23.0}},
  'mistralai/Mistral-7B-Instruct-v0.3': {'bug': {'precision': 0.5454545454545454,
    'recall': 0.6666666666666666,
    'f1-score': 0.6,
    'support': 18.0},
   'feature': {'precision': 0.2608695652173913,
    'recall': 0.

In [ ]:
MODEL_DISPLAY = {
    "google/gemma-3-12b-it": "Gemma-3-12B",
    "meta-llama/Llama-3.1-8B-Instruct": "Llama-3.1-8B",
    "mistralai/Mistral-7B-Instruct-v0.3": "Mistral-7B",
    "Qwen/Qwen2.5-7B-Instruct": "Qwen2.5-7B",
}

# ============================================================
# FORMAT FUNCTION
# ============================================================

def fmt(x):
    return f"{x:.2f}"


# ============================================================
# GENERATE LATEX
# ============================================================

latex = []

latex.append("\\begin{table*}[t]")
latex.append("\\centering")
latex.append("\\caption{Per-label precision, recall, and F1 for zero-shot predictions across repositories and models.}")
latex.append("\\label{tab:per-label-zero-shot}")
latex.append("\\resizebox{\\textwidth}{!}{")
latex.append("\\begin{tabular}{ll|ccc|ccc|ccc}")
latex.append("\\toprule")
latex.append("\\textbf{Repository} & \\textbf{Model} & \\multicolumn{3}{c|}{\\textbf{Bug}} & \\multicolumn{3}{c|}{\\textbf{Feature}} & \\multicolumn{3}{c}{\\textbf{Question}} \\\\")
latex.append("\\cmidrule(lr){3-5} \\cmidrule(lr){6-8} \\cmidrule(lr){9-11}")
latex.append("& & P & R & F1 & P & R & F1 & P & R & F1 \\\\")
latex.append("\\midrule")

for repo in REPOS:

    display_repo = repo.split("_")[-1].capitalize() if "_" in repo else repo

    models = list(results[repo].keys())

    for i, model in enumerate(models):

        bug = results[repo][model]["bug"]
        feat = results[repo][model]["feature"]
        ques = results[repo][model]["question"]

        row = []

        if i == 0:
            row.append(f"\\multirow{{{len(models)}}}{{*}}{{{display_repo}}}")
        else:
            row.append("")

        row.append(MODEL_DISPLAY[model])

        row += [
            fmt(bug.get("precision", 0)),
            fmt(bug.get("recall", 0)),
            fmt(bug.get("f1-score", 0)),

            fmt(feat.get("precision", 0)),
            fmt(feat.get("recall", 0)),
            fmt(feat.get("f1-score", 0)),

            fmt(ques.get("precision", 0)),
            fmt(ques.get("recall", 0)),
            fmt(ques.get("f1-score", 0)),
        ]

        latex.append(" & ".join(row) + " \\\\")

    latex.append("\\midrule")

latex.append("\\bottomrule")
latex.append("\\end{tabular}}")
latex.append("\\end{table*}")

print("\n".join(latex))

\begin{table*}[t]
\centering
\caption{Per-label precision, recall, and F1 for zero-shot predictions across repositories and models.}
\label{tab:per-label-zero-shot}
\resizebox{\textwidth}{!}{
\begin{tabular}{ll|ccc|ccc|ccc}
\toprule
\textbf{Repository} & \textbf{Model} & \multicolumn{3}{c|}{\textbf{Bug}} & \multicolumn{3}{c|}{\textbf{Feature}} & \multicolumn{3}{c}{\textbf{Question}} \\
\cmidrule(lr){3-5} \cmidrule(lr){6-8} \cmidrule(lr){9-11}
& & P & R & F1 & P & R & F1 & P & R & F1 \\
\midrule
\multirow{4}{*}{React} & Gemma-3-12B & 0.56 & 1.00 & 0.72 & 0.80 & 0.50 & 0.62 & 0.83 & 0.43 & 0.57 \\
 & Llama-3.1-8B & 0.37 & 0.94 & 0.53 & 0.50 & 0.12 & 0.20 & 1.00 & 0.04 & 0.08 \\
 & Mistral-7B & 0.55 & 0.67 & 0.60 & 0.26 & 0.75 & 0.39 & 1.00 & 0.17 & 0.30 \\
 & Qwen2.5-7B & 0.59 & 0.89 & 0.71 & 0.55 & 0.75 & 0.63 & 0.91 & 0.43 & 0.59 \\
\midrule
\multirow{4}{*}{Bitcoin} & Gemma-3-12B & 0.80 & 0.98 & 0.88 & 0.89 & 0.61 & 0.72 & 0.64 & 0.44 & 0.52 \\
 & Llama-3.1-8B & 0.64 & 1.00 & 0.78 & 0.

In [21]:
import json
import os
import numpy as np
import pandas as pd

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# ============================================================
# CONFIG
# ============================================================

DATA_DIR = "../data"
EMBED_DIR = "../data/test_embeddings"
os.makedirs(EMBED_DIR, exist_ok=True)

REPOS = [
    "facebook_react",
    "bitcoin_bitcoin",
    "opencv_opencv",
    "tensorflow_tensorflow",
    "microsoft_vscode",
]

# ============================================================
# LOAD EMBEDDING MODEL
# ============================================================

print("Loading embedding model...")
model = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")

# ============================================================
# HELPER FUNCTIONS
# ============================================================

def load_repo(repo):
    test_path = f"{DATA_DIR}/{repo}/{repo}_test_issues_normalized.json"

    # load test
    with open(test_path, "r", encoding="utf-8") as f:
        test_data = json.load(f)
        
    data = test_data

    df = pd.DataFrame(data)

    # normalize label
    df["labels"] = df["labels"].str.strip().str.lower()

    # combine title + body
    df["text"] = df["title"].fillna("") + " " + df["body"].fillna("")
    
    print("\nLabel distribution:")
    print(df["labels"].value_counts())

    return df[["text", "labels"]]


def compute_similarity(df, repo):

    embed_path = f"{EMBED_DIR}/{repo}_embeddings.npy"

    texts = df["text"].tolist()

    # ------------------------------------------------
    # LOAD OR COMPUTE EMBEDDINGS
    # ------------------------------------------------
    if os.path.exists(embed_path):
        print(f"Loading cached embeddings for {repo}...")
        embeddings = np.load(embed_path)

    else:
        print(f"Encoding {len(texts)} issues...")

        embeddings = model.encode(
            texts,
            show_progress_bar=True,
            convert_to_numpy=True,
            normalize_embeddings=True
        )

        np.save(embed_path, embeddings)
        print(f"Saved embeddings to {embed_path}")

    # attach embeddings to dataframe
    df["embedding"] = list(embeddings)

    labels = sorted(df["labels"].unique())

    rows = []

    # ------------------------------------------------
    # COMPUTE CROSS-LABEL SIMILARITY
    # ------------------------------------------------
    for i in range(len(labels)):
        for j in range(i + 1, len(labels)):

            label_a = labels[i]
            label_b = labels[j]

            emb_a = np.vstack(df[df["labels"] == label_a]["embedding"])
            emb_b = np.vstack(df[df["labels"] == label_b]["embedding"])

            sim = cosine_similarity(emb_a, emb_b).mean()

            rows.append({
                "repo": repo,
                "label_a": label_a,
                "label_b": label_b,
                "similarity": sim
            })

    return rows
# ============================================================
# MAIN LOOP
# ============================================================

results = []

for repo in REPOS:

    print("\n===================================================")
    print(f"Processing repository: {repo}")
    print("===================================================")

    df = load_repo(repo)

    rows = compute_similarity(df, repo)
    results.extend(rows)


# ============================================================
# SAVE RESULTS
# ============================================================

results_df = pd.DataFrame(results)
results_df

Loading embedding model...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Processing repository: facebook_react

Label distribution:
labels
question    23
bug         18
feature      8
Name: count, dtype: int64
Encoding 49 issues...


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/5 [00:00<?, ?it/s]

Saved embeddings to ../data/embeddings/tensorflow_tensorflow_embeddings.npy

Processing repository: microsoft_vscode

Label distribution:
labels
question    301
feature      90
bug          27
Name: count, dtype: int64
Encoding 418 issues...


Batches:   0%|          | 0/14 [00:00<?, ?it/s]

Saved embeddings to ../data/embeddings/microsoft_vscode_embeddings.npy


,repo,label_a,label_b,similarity
0,facebook_react,bug,feature,0.329502
1,facebook_react,bug,question,0.305516
2,facebook_react,feature,question,0.284530
3,bitcoin_bitcoin,bug,feature,0.092678
4,bitcoin_bitcoin,bug,question,0.136533
5,bitcoin_bitcoin,feature,question,0.074093
6,opencv_opencv,bug,feature,0.241150
7,opencv_opencv,bug,question,0.365528
8,opencv_opencv,feature,question,0.227332
9,tensorflow_tensorflow,bug,feature,0.303039


In [22]:
repo_names = {
    "facebook_react": "React",
    "bitcoin_bitcoin": "Bitcoin",
    "opencv_opencv": "Opencv",
    "tensorflow_tensorflow": "Tensorflow",
    "microsoft_vscode": "Vscode"
}

print(r"\begin{table}[ht]")
print(r"  \fontsize{8}{9}\selectfont")
print(r"  \tabcolsep=0.05cm")
print(r"\centering")
print(r"\caption{Label Embedding Similarity Across Repositories}")
print(r"\label{tab:label_similarity}")
print(r"\renewcommand{\arraystretch}{1.2}")
print(r"\setlength{\tabcolsep}{2pt}")
print(r"\begin{tabular}{llll}")
print(r"\hline")
print(r"\textbf{Repository} & \textbf{Label A} & \textbf{Label B} & \textbf{Similarity} \\ \hline")

for repo, group in results_df.groupby("repo"):
    first = True
    for _, row in group.iterrows():
        repo_name = repo_names.get(repo, repo) if first else ""
        first = False
        print(f"{repo_name} & {row['label_a'].capitalize()} & {row['label_b'].capitalize()} & {row['similarity']:.3f} \\\\")
    print(r"\hline")

print(r"\end{tabular}")
print(r"\end{table}")

\begin{table}[ht]
  \fontsize{8}{9}\selectfont
  \tabcolsep=0.05cm
\centering
\caption{Label Embedding Similarity Across Repositories}
\label{tab:label_similarity}
\renewcommand{\arraystretch}{1.2}
\setlength{\tabcolsep}{2pt}
\begin{tabular}{llll}
\hline
\textbf{Repository} & \textbf{Label A} & \textbf{Label B} & \textbf{Similarity} \\ \hline
Bitcoin & Bug & Feature & 0.093 \\
 & Bug & Question & 0.137 \\
 & Feature & Question & 0.074 \\
\hline
React & Bug & Feature & 0.330 \\
 & Bug & Question & 0.306 \\
 & Feature & Question & 0.285 \\
\hline
Vscode & Bug & Feature & 0.190 \\
 & Bug & Question & 0.245 \\
 & Feature & Question & 0.193 \\
\hline
Opencv & Bug & Feature & 0.241 \\
 & Bug & Question & 0.366 \\
 & Feature & Question & 0.227 \\
\hline
Tensorflow & Bug & Feature & 0.303 \\
 & Bug & Question & 0.341 \\
 & Feature & Question & 0.270 \\
\hline
\end{tabular}
\end{table}


In [ ]:
import os
import json
import numpy as np
import pandas as pd

from sentence_transformers import SentenceTransformer


# ============================================================
# CONFIG
# ============================================================

DATA_DIR = "../data"
EMBEDDINGS_DIR = "../embeddings"

REPOS = [
    "facebook_react",
    "bitcoin_bitcoin",
    "opencv_opencv",
    "tensorflow_tensorflow",
    "microsoft_vscode",
]

DATA_SPLIT = "train"  

MAX_SAMPLES_PER_LABEL = 7000
RANDOM_SEED = 42

os.makedirs(EMBEDDINGS_DIR, exist_ok=True)


# ============================================================
# LOAD EMBEDDING MODEL
# ============================================================

print("Loading embedding model...")
model = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")


# ============================================================
# LOAD DATA
# ============================================================

def load_repo(repo, split):

    if split == "test":
        path = f"{DATA_DIR}/{repo}/{repo}_test_issues_normalized.json"
    elif split == "train":
        path = f"{DATA_DIR}/{repo}/{repo}_train_issues_normalized.json"
    else:
        raise ValueError("Unknown split")

    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    df = pd.DataFrame(data)

    df["label"] = df["labels"].str.strip().str.lower()
    df["text"] = df["title"].fillna("") + " " + df["body"].fillna("")

    return df[["text", "label"]]


# ============================================================
# EMBEDDING CACHE
# ============================================================

def get_embeddings(repo, label, texts, split):

    cache_file = f"{EMBEDDINGS_DIR}/{repo}_{split}_{label}_embeddings.npy"

    if os.path.exists(cache_file):

        print(f"Loading cached embeddings → {cache_file}")
        embeddings = np.load(cache_file)

    else:

        print(f"{repo} | {label} | encoding {len(texts)} issues")

        embeddings = model.encode(
            texts,
            show_progress_bar=False,
            convert_to_numpy=True,
            normalize_embeddings=True,
        )

        np.save(cache_file, embeddings)

    return embeddings


# ============================================================
# WITHIN-LABEL SIMILARITY
# ============================================================

def compute_within_label_similarity(df, repo, split):

    results = []

    labels = sorted(df["label"].unique())

    for label in labels:

        subset = df[df["label"] == label]

        # sample if dataset is large
        if len(subset) > MAX_SAMPLES_PER_LABEL:
            subset = subset.sample(MAX_SAMPLES_PER_LABEL, random_state=RANDOM_SEED)

        texts = subset["text"].tolist()

        embeddings = get_embeddings(repo, label, texts, split)

        # cosine similarity (faster since embeddings normalized)
        sim_matrix = embeddings @ embeddings.T

        # remove self similarity
        mask = ~np.eye(sim_matrix.shape[0], dtype=bool)

        mean_similarity = sim_matrix[mask].mean()

        results.append({
            "repo": repo,
            "label": label,
            "mean_similarity": mean_similarity,
            "num_issues": len(texts),
        })

    return results


# ============================================================
# MAIN
# ============================================================

all_results = []

for repo in REPOS:

    print("\n===================================================")
    print(f"Processing {repo}")
    print("===================================================")

    df = load_repo(repo, DATA_SPLIT)

    results = compute_within_label_similarity(df, repo, DATA_SPLIT)

    all_results.extend(results)


results_df = pd.DataFrame(all_results)

print("\nFinal Results:")
results_df

Loading embedding model...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Processing facebook_react
facebook_react | bug | encoding 143 issues
facebook_react | feature | encoding 69 issues
facebook_react | question | encoding 176 issues

Processing bitcoin_bitcoin
bitcoin_bitcoin | bug | encoding 487 issues
bitcoin_bitcoin | feature | encoding 225 issues
bitcoin_bitcoin | question | encoding 126 issues

Processing opencv_opencv
opencv_opencv | bug | encoding 136 issues
opencv_opencv | feature | encoding 82 issues
opencv_opencv | question | encoding 826 issues

Processing tensorflow_tensorflow
tensorflow_tensorflow | bug | encoding 480 issues


In [7]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path("../data")
ZERO_SHOT_DIR = Path("../rqs/zero_shot")

REPOS = [
    "facebook_react",
    "bitcoin_bitcoin",
    "opencv_opencv",
    "tensorflow_tensorflow",
    "microsoft_vscode",
]

MODELS = [
    "google/gemma-3-12b-it",
    "mistralai/Mistral-7B-Instruct-v0.3",
    "meta-llama/Llama-3.1-8B-Instruct",
    "Qwen/Qwen2.5-7B-Instruct"
]

rows = []

for repo in REPOS:

    for model in MODELS:

        provider, model_name = model.split("/", 1)

        pred_dir = (
            ZERO_SHOT_DIR
            / repo
            / provider
            / model_name
            / "logs"
            / "generations"
            / "zero_shot"
            / "run_0"
        )

        if not pred_dir.exists():
            continue

        prediction_files = list(pred_dir.glob("predictions_*.csv"))

        for file in prediction_files:

            print(f"Processing {file}")

            df = pd.read_csv(file)

            df_filtered = df[
                (df["Ground_Truth"] == "question") &
                (df["Prediction"] == "bug")
            ]

            df_filtered = df_filtered[["number", "Title", "Body", "Ground_Truth", "Prediction"]]
            df_filtered["repo"] = repo
            df_filtered["model"] = model

            rows.append(df_filtered)

# Combine everything
all_errors = pd.concat(rows, ignore_index=True)

# Count how many models made this mistake per issue
counts = (
    all_errors
    .groupby(["repo", "number"])
    .model
    .nunique()
    .reset_index(name="model_count")
)

# Keep only issues where ALL models made the same error
target = counts[counts["model_count"] == len(MODELS)]

# Merge back to get issue info
final = all_errors.merge(target[["repo", "number"]], on=["repo", "number"])

# Keep only one row per repo/issue
final = final.drop_duplicates(subset=["repo", "number"])

final = final[[
    "repo",
    "number",
    "Title",
    "Body",
    "Ground_Truth",
    "Prediction"
]]

output_path = ZERO_SHOT_DIR / "question_predicted_bug_all_models.csv"
final.to_csv(output_path, index=False)

print(final)
print(f"\nSaved to {output_path}")

Processing ..\rqs\zero_shot\facebook_react\google\gemma-3-12b-it\logs\generations\zero_shot\run_0\predictions_dda78b9e-fb7e-4b43-b3af-15b1fc342ac6.csv
Processing ..\rqs\zero_shot\facebook_react\mistralai\Mistral-7B-Instruct-v0.3\logs\generations\zero_shot\run_0\predictions_ff821366-682f-4b4b-9dc5-be0606d74da9.csv
Processing ..\rqs\zero_shot\facebook_react\meta-llama\Llama-3.1-8B-Instruct\logs\generations\zero_shot\run_0\predictions_6eaca854-09f9-4fbb-81fe-9f41e3cf6909.csv
Processing ..\rqs\zero_shot\facebook_react\Qwen\Qwen2.5-7B-Instruct\logs\generations\zero_shot\run_0\predictions_1f2a41dc-f315-4df5-a821-face326ef643.csv
Processing ..\rqs\zero_shot\bitcoin_bitcoin\google\gemma-3-12b-it\logs\generations\zero_shot\run_0\predictions_650984cd-7e85-4068-96e3-08c9f82a20bf.csv
Processing ..\rqs\zero_shot\bitcoin_bitcoin\mistralai\Mistral-7B-Instruct-v0.3\logs\generations\zero_shot\run_0\predictions_f21c1873-a591-4d4e-aab2-77f42f6ce0fd.csv
Processing ..\rqs\zero_shot\bitcoin_bitcoin\meta-lla